In [ ]:
# Modules Installation
!pip install -q groq

In [ ]:
# Modules Import
import os
import datetime
import time
import json

from groq import Groq

In [ ]:
##############
# Prompts
##############
# Instruction Only Prompt
prompt_1 = '''----
You are an expert linguist assisting with the Relationship Extraction task. The task is to find out the relationship between the head entity and the tail entity in the given sentence. The assigned relationship must be in (CAUSE-EFFECT, COMPONENT-WHOLE, CONTENT-CONTAINER, ENTITY-DESTINATION, ENTITY-ORIGIN, INSTRUMENT-AGENCY, MEMBER-COLLECTION, MESSAGE-TOPIC, PRODUCT-PRODUCER, NONE).
When the relationship between the head entity and the tail entity cannot be identified from one of the relationships listed above, you must assign the relationship as NONE.
You will generate a response in the form: In the given sentence, the relationship between <head-entity> and <tail-entity> entities is: <best from the list of relationships>.
----
Sentence: {}
In the given sentence, the relationship between {} and {} entities is:'''

# Instruction + Relationship Description Prompt
prompt_2 = '''----
You are an expert linguist assisting with the Relationship Extraction task. The task is to find out the relationship between the head entity and the tail entity in the given sentence. The assigned relationship must be in (CAUSE-EFFECT, COMPONENT-WHOLE, CONTENT-CONTAINER, ENTITY-DESTINATION, ENTITY-ORIGIN, INSTRUMENT-AGENCY, MEMBER-COLLECTION, MESSAGE-TOPIC, PRODUCT-PRODUCER, NONE).
----
An explaination for each Relationship is given below:
1. CAUSE-EFFECT relationship occurs when an event or object is considered a cause for an effect. 
2. COMPONENT-WHOLE relationship occurs when an object has an operational use and is part of a bigger whole.
3. CONTENT-CONTAINER relationship occurs when an object is physically stored or carried in a container. 
4. ENTITY-DESTINATION relationship occurs when an entity is moving towards a destination. 
5. ENTITY-ORIGIN relationship occurs when an entity is coming from or is derived from an origin (position or material). 
6. INSTRUMENT-AGENCY relationship occurs when an agent uses an instrument. 
7. MEMBER-COLLECTION relationship occurs when a member is a non-functional part of a collection. 
8. MESSAGE-TOPIC relationship occurs when a communication message contains information about a topic. 
9. PRODUCT-PRODUCER relationship occurs when a producer is actively involved in the creation of the product. 
10. NONE relationship occurs when the relationship between the head entity and the tail entity cannot be categorised in one of the above nine relationships.
---
You will generate response in the form: In the given sentence, the relationship between <head-entity> and <tail-entity> entities is: <best from the list of relationships>.
----
Sentence: {}
In the given sentence, the relationship between {} and {} entities is:'''

# Instruction + 5-shot Example Demonstrations Prompt
prompt_3 = '''----
You are an expert linguist assisting with the Relationship Extraction task. The task is to find out the relationship between the head entity and the tail entity in the given sentence. The assigned relationship must be in (CAUSE-EFFECT, COMPONENT-WHOLE, CONTENT-CONTAINER, ENTITY-DESTINATION, ENTITY-ORIGIN, INSTRUMENT-AGENCY, MEMBER-COLLECTION, MESSAGE-TOPIC, PRODUCT-PRODUCER, NONE). 
When the relationship between the head entity and the tail entity cannot be identified from one of the relationships listed above, you must assign the relationship as NONE.
You will generate response in the form: In the given sentence, the relationship between <head-entity> and <tail-entity> entities is: <best from the list of relationships>. 
----
Here are some examples of Relationship Extraction task:
Example-1:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
Example-2:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
Example-3:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
Example-4:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
Example-5:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
----
Based on above instructions, you need to identify relationship between head and tail entity in the below sentence.
Sentence: {}
In the given sentence, the relationship between {} and {} entities is:'''

# Instruction + 10-shot Example Demonstrations Prompt
prompt_3a = '''----
You are an expert linguist assisting with the Relationship Extraction task. The task is to find out the relationship between the head entity and the tail entity in the given sentence. The assigned relationship must be in (CAUSE-EFFECT, COMPONENT-WHOLE, CONTENT-CONTAINER, ENTITY-DESTINATION, ENTITY-ORIGIN, INSTRUMENT-AGENCY, MEMBER-COLLECTION, MESSAGE-TOPIC, PRODUCT-PRODUCER, NONE). 
When the relationship between the head entity and the tail entity cannot be identified from one of the relationships listed above, you must assign the relationship as NONE.
You will generate response in the form: In the given sentence, the relationship between <head-entity> and <tail-entity> entities is: <best from the list of relationships>. 
----
Here are some examples of Relationship Extraction task:
Example-1:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
Example-2:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
Example-3:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
Example-4:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
Example-5:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
Example-6:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
Example-7:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
Example-8:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
Example-9:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
Example-10:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
----
Based on above instructions, you need to identify relationship between head and tail entity in the below sentence.
Sentence: {}
In the given sentence, the relationship between {} and {} entities is:'''

# Instruction + 5-shot CoT Demonstrations Prompt
prompt_4 = '''----
You are an expert linguist assisting with the Relationship Extraction task. The task is to find out the relationship between the head entity and the tail entity in the given sentence. The assigned relationship must be in (CAUSE-EFFECT, COMPONENT-WHOLE, CONTENT-CONTAINER, ENTITY-DESTINATION, ENTITY-ORIGIN, INSTRUMENT-AGENCY, MEMBER-COLLECTION, MESSAGE-TOPIC, PRODUCT-PRODUCER, NONE). 
When the relationship between the head entity and the tail entity cannot be identified from one of the relationships listed above, you must assign the relationship as NONE.
You will generate response in the form: In the given sentence, the relationship between <head-entity> and <tail-entity> entities is: <best from the list of relationships>. 
----
Here are some examples with Chain of Thoughts (COT) for the Relationship Extraction task:
Example-1:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
COT Exaplanation: {}
Example-2:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
COT Exaplanation: {}
Example-3:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
COT Exaplanation: {}
Example-4:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
COT Exaplanation: {}
Example-5:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
COT Exaplanation: {}
----
Sentence: {}
In the given sentence, the relationship between {} and {} entities is:'''

# Instruction + 10-shot CoT Demonstrations Prompt
prompt_4a = '''----
You are an expert linguist assisting with the Relationship Extraction task. The task is to find out the relationship between the head entity and the tail entity in the given sentence. The assigned relationship must be in (CAUSE-EFFECT, COMPONENT-WHOLE, CONTENT-CONTAINER, ENTITY-DESTINATION, ENTITY-ORIGIN, INSTRUMENT-AGENCY, MEMBER-COLLECTION, MESSAGE-TOPIC, PRODUCT-PRODUCER, NONE). 
When the relationship between the head entity and the tail entity cannot be identified from one of the relationships listed above, you must assign the relationship as NONE.
You will generate response in the form: In the given sentence, the relationship between <head-entity> and <tail-entity> entities is: <best from the list of relationships>. 
----
Here are some examples with Chain of Thoughts (COT) for the Relationship Extraction task:
Example-1:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
COT Exaplanation: {}
Example-2:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
COT Exaplanation: {}
Example-3:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
COT Exaplanation: {}
Example-4:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
COT Exaplanation: {}
Example-5:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
COT Exaplanation: {}
Example-6:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
COT Exaplanation: {}
Example-7:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
COT Exaplanation: {}
Example-8:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
COT Exaplanation: {}
Example-9:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
COT Exaplanation: {}
Example-10:
Sentence: {}
In the above sentence, the relationship between {} and {} entities is: {}
COT Exaplanation: {}
----
Sentence: {}
In the given sentence, the relationship between {} and {} entities is:'''

In [ ]:
# Configurations
dataset = "semeval2010-task8-dataset"
dataset_dir = "/kaggle/input/semeval2010-task8-dataset"

#model_name = "gemma2-9b-it"
#model_name = "llama-3.3-70b-versatile"
#model_name = "meta-llama/llama-4-maverick-17b-128e-instruct"
#model_name = "qwen-2.5-32b"
model_name = "qwen/qwen3-32b"

prompt_ver = "prompt_4a"
prompt = eval(prompt_ver)

config = { }
config['dataset'] = dataset
config['model_name'] = model_name
config['temperature'] = 0.5
config['top_p'] = 0.90
config['max_new_tokens'] = 200
config['stream'] = False
config['num_return_sequences'] = 1
config['prompt_ver'] = prompt_ver
config['prompt'] = prompt

dt_tm = datetime.datetime.now()
config['date_time'] = dt_tm.strftime("%Y-%m-%d_%H-%M")

In [ ]:
# Save configuration in config.json
output_path = "/kaggle/working"

if "/" in model_name:
    model_name = model_name.split("/")[1]
print(model_name)
print(config['model_name'])
    
path_model_name = os.path.join(output_path, model_name)
path_prompt_ver = prompt_ver.replace("_", "-")
path_temp = "T" + str(config['temperature']).replace(".","")

output_folder = path_model_name + "_" + path_prompt_ver + "_" + path_temp 

# Create folder with model name, temperature information
output_path = os.path.join(output_path, output_folder) 
if not os.path.exists(output_path):
    os.makedirs(output_path)
    
# Write Configureation to config.json file
config_file_name = os.path.join(output_path, "config.json")
with open(config_file_name, 'w') as fp_config:
    json.dump(config, fp_config)
    fp_config.close()

In [ ]:
client = Groq(
    api_key="GROQ_API_KEY",
)

In [ ]:
def get_examples(file):
    # open the JSON file
    with open(file) as f: 
        # read all json objects as example dictionaries and store in the example list
        examples_list = json.load(f)

        # Close the file
        f.close()

    # return list of examples: each example is dictionary
    return examples_list

In [ ]:
# Dataset initialization
dataset_testfile = os.path.join(dataset_dir, "test.json")
dataset_trainfile = os.path.join(dataset_dir, "train.json")

list_test_examples = get_examples(dataset_testfile)
list_train_examples = get_examples(dataset_trainfile)

In [ ]:
def get_fewshot_examples(test_example, fewshots = 5):
    
    #examples_indices = random.sample(range(0, len(list_train_examples)), fewshots)                # for Random retrieval of demonstrations
    examples_indices = [ex_id-1 for ex_id in test_example['similar_sentences']][0:fewshots]      # for Similarity-based retrieval of demonstrations

    fs_examples = []
    for index in examples_indices:
        fs_examples.append(list_train_examples[index])

    return fs_examples

In [ ]:
def prepare_prompt(test_example):
    
    if prompt_ver == "prompt_1" or prompt_ver == "prompt_2":
        prompt_text = prompt.format(example['sentence'], example['e1'], example['e2'])
    
    elif prompt_ver == "prompt_3":
        fs_examples = get_fewshot_examples(test_example)
        prompt_text = prompt.format(
                    fs_examples[0]['sentence'], fs_examples[0]['e1'], fs_examples[0]['e2'], fs_examples[0]['relation'],
                    fs_examples[1]['sentence'], fs_examples[1]['e1'], fs_examples[1]['e2'], fs_examples[1]['relation'],
                    fs_examples[2]['sentence'], fs_examples[2]['e1'], fs_examples[2]['e2'], fs_examples[2]['relation'],
                    fs_examples[3]['sentence'], fs_examples[3]['e1'], fs_examples[3]['e2'], fs_examples[3]['relation'],
                    fs_examples[4]['sentence'], fs_examples[4]['e1'], fs_examples[4]['e2'], fs_examples[4]['relation'],
                    example['sentence'], example['e1'], example['e2'])

    elif prompt_ver == "prompt_3a":
        fs_examples = get_fewshot_examples(test_example, fewshots = 10)
        prompt_text = prompt.format(
                    fs_examples[0]['sentence'], fs_examples[0]['e1'], fs_examples[0]['e2'], fs_examples[0]['relation'],
                    fs_examples[1]['sentence'], fs_examples[1]['e1'], fs_examples[1]['e2'], fs_examples[1]['relation'],
                    fs_examples[2]['sentence'], fs_examples[2]['e1'], fs_examples[2]['e2'], fs_examples[2]['relation'],
                    fs_examples[3]['sentence'], fs_examples[3]['e1'], fs_examples[3]['e2'], fs_examples[3]['relation'],
                    fs_examples[4]['sentence'], fs_examples[4]['e1'], fs_examples[4]['e2'], fs_examples[4]['relation'],
                    fs_examples[5]['sentence'], fs_examples[5]['e1'], fs_examples[5]['e2'], fs_examples[5]['relation'],
                    fs_examples[6]['sentence'], fs_examples[6]['e1'], fs_examples[6]['e2'], fs_examples[6]['relation'],
                    fs_examples[7]['sentence'], fs_examples[7]['e1'], fs_examples[7]['e2'], fs_examples[7]['relation'],
                    fs_examples[8]['sentence'], fs_examples[8]['e1'], fs_examples[8]['e2'], fs_examples[8]['relation'],
                    fs_examples[9]['sentence'], fs_examples[9]['e1'], fs_examples[9]['e2'], fs_examples[9]['relation'],
                    example['sentence'], example['e1'], example['e2'])
    
    elif prompt_ver == "prompt_4":
        fs_examples = get_fewshot_examples(test_example)
        prompt_text = prompt.format(
                    fs_examples[0]['sentence'], fs_examples[0]['e1'], fs_examples[0]['e2'], fs_examples[0]['relation'], fs_examples[0]['cot_explanation'],
                    fs_examples[1]['sentence'], fs_examples[1]['e1'], fs_examples[1]['e2'], fs_examples[1]['relation'], fs_examples[1]['cot_explanation'],
                    fs_examples[2]['sentence'], fs_examples[2]['e1'], fs_examples[2]['e2'], fs_examples[2]['relation'], fs_examples[2]['cot_explanation'],
                    fs_examples[3]['sentence'], fs_examples[3]['e1'], fs_examples[3]['e2'], fs_examples[3]['relation'], fs_examples[3]['cot_explanation'],
                    fs_examples[4]['sentence'], fs_examples[4]['e1'], fs_examples[4]['e2'], fs_examples[4]['relation'], fs_examples[4]['cot_explanation'],
                    example['sentence'], example['e1'], example['e2'])
    
    elif prompt_ver == "prompt_4a":
        fs_examples = get_fewshot_examples(test_example, fewshots = 10)
        prompt_text = prompt.format(
                    fs_examples[0]['sentence'], fs_examples[0]['e1'], fs_examples[0]['e2'], fs_examples[0]['relation'], fs_examples[0]['cot_explanation'],
                    fs_examples[1]['sentence'], fs_examples[1]['e1'], fs_examples[1]['e2'], fs_examples[1]['relation'], fs_examples[1]['cot_explanation'],
                    fs_examples[2]['sentence'], fs_examples[2]['e1'], fs_examples[2]['e2'], fs_examples[2]['relation'], fs_examples[2]['cot_explanation'],
                    fs_examples[3]['sentence'], fs_examples[3]['e1'], fs_examples[3]['e2'], fs_examples[3]['relation'], fs_examples[3]['cot_explanation'],
                    fs_examples[4]['sentence'], fs_examples[4]['e1'], fs_examples[4]['e2'], fs_examples[4]['relation'], fs_examples[4]['cot_explanation'],
                    fs_examples[5]['sentence'], fs_examples[5]['e1'], fs_examples[5]['e2'], fs_examples[5]['relation'], fs_examples[5]['cot_explanation'],
                    fs_examples[6]['sentence'], fs_examples[6]['e1'], fs_examples[6]['e2'], fs_examples[6]['relation'], fs_examples[6]['cot_explanation'],
                    fs_examples[7]['sentence'], fs_examples[7]['e1'], fs_examples[7]['e2'], fs_examples[7]['relation'], fs_examples[7]['cot_explanation'],
                    fs_examples[8]['sentence'], fs_examples[8]['e1'], fs_examples[8]['e2'], fs_examples[8]['relation'], fs_examples[8]['cot_explanation'],
                    fs_examples[9]['sentence'], fs_examples[9]['e1'], fs_examples[9]['e2'], fs_examples[9]['relation'], fs_examples[9]['cot_explanation'],
                    example['sentence'], example['e1'], example['e2'])
    else:
        prompt_text = "none"

    return prompt_text

In [ ]:
#### Run model and extract results

# Open output file to store results
file_name = os.path.join(output_path, 
            "result_" + model_name + "_" + dataset + "1.json")
fp1 = open(file_name, 'w')

print("Processing Examples: ")
for example in list_test_examples:
    
    time.sleep(0.2)
    prompt_text = prepare_prompt(example)
    
    try:
        chat_completion = client.chat.completions.create(
                                messages=[
                                    {
                                        "role": "user",
                                        "content": prompt_text,
                                    }
                                ],
                                model=config['model_name'],
                                temperature=config['temperature'],
                                max_tokens=config['max_new_tokens'],
                                top_p=config['top_p'],
                                stream=config['stream'])

        result = chat_completion.choices[0].message.content
    except:
        print("Exception Occured")
        fp1.close()
        
    example['output'] = chat_completion.choices[0].message.content
    
    json.dump(example, fp1)
    print("Example: {id:6s} {e1:20s} {e2:20s} {rel:25s}".format(id=example['id'], e1=example['e1'], 
                    e2=example['e2'], rel=example['relation']))
    

fp1.close()

In [ ]:
# Save output dictionary in a file.
file_name = os.path.join(output_path, 
            "result_" + model_name + "_" + dataset + ".json")

with open(file_name, 'w') as fp:
    json.dump(list_test_examples, fp)
    fp.close()

In [ ]:
print("**** THE-END ****")